# FINPLE Monthly Metrics One-Click Pipeline

Step 114-2C offline source adapter Colab workflow. The notebook exposes five operator sections and delegates calculation, source adapter selection, raw daily normalization, and audit generation to the repository module. In a fresh Colab runtime, upload the Step 114-2C execution package ZIP when prompted. No private GitHub token is required.

## 1. Check settings

Confirm CONFIG only. The bootstrap first looks for a repository checkout and otherwise asks for an uploaded execution package ZIP. Step 114-2C remains offline-only and supports fixture, manual upload CSV, and synthetic public-source fixture adapter modes.

In [ ]:
from pathlib import Path
import sys
import zipfile

def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "scripts" / "metrics_pipeline").exists() and (candidate / "data" / "fixtures" / "monthly-metrics").exists():
            return candidate
    return None

REPO_ROOT = find_repo_root(Path.cwd())
if REPO_ROOT is None:
    try:
        from google.colab import files
        print("Upload the Step 114-2B execution package ZIP containing scripts/metrics_pipeline and data/fixtures/monthly-metrics.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No execution package ZIP uploaded.")
        zip_name = next(iter(uploaded))
        extract_root = Path("/content/finple_step114_2c_execution_package")
        extract_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_name) as archive:
            archive.extractall(extract_root)
        REPO_ROOT = find_repo_root(extract_root)
    except ImportError as exc:
        raise RuntimeError("Repository files were not found. In fresh Colab, upload the Step 114-2B execution package ZIP.") from exc
if REPO_ROOT is None:
    raise RuntimeError("Execution package ZIP did not contain scripts/metrics_pipeline and data/fixtures/monthly-metrics.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CONFIG = {
    "metric_base_date": "2026-06-30",
    "market_scope": ["US", "KR"],
    "selected_cagr_policy": "rolling_median_all_markets",
    "current_price_display": False,
    "total_return_cagr_mode": "reference_only",
    "output_version": "2026_06",
    "input_mode": "fixture",  # fixture | manual_upload | public_source_fixture
    "input_dir": str(REPO_ROOT / "data" / "fixtures" / "monthly-metrics"),
    "output_dir": str(REPO_ROOT / "outputs" / "monthly-metrics" / "2026_06"),
    "deterministic_fixture": True,
    "random_seed": 1142,
}

print(f"Repository root: {REPO_ROOT}")
CONFIG

## 2. Check inputs

Fixture mode uses committed offline files only. No external market-data provider is called.

In [ ]:
input_dir = Path(CONFIG["input_dir"])
required_inputs = ["candidates.csv", "benchmark_map.csv", "monthly_prices.csv"]
if CONFIG["input_mode"] == "fixture":
    required_inputs.append("raw_daily_prices.csv")
elif CONFIG["input_mode"] == "manual_upload":
    required_inputs.append(CONFIG.get("manual_upload_file", "manual_upload_raw_daily_prices.csv"))
elif CONFIG["input_mode"] == "public_source_fixture":
    required_inputs.append(CONFIG.get("public_source_fixture_file", "public_source_fixture_prices.csv"))
else:
    raise ValueError(f"Unsupported offline input_mode: {CONFIG['input_mode']}")
for file_name in required_inputs:
    path = input_dir / file_name
    print(f"{file_name}: {'OK' if path.exists() else 'MISSING'}")

## 3. Run pipeline

Run the single repository entrypoint.

In [ ]:
from scripts.metrics_pipeline import run_finple_monthly_metrics_pipeline

RESULT = run_finple_monthly_metrics_pipeline(CONFIG)

## 4. Review summary

Review counts, adapter summary, manifest versions, and generated paths before using any output.

In [ ]:
print("FINPLE Monthly Metrics Pipeline Complete")
print(f"Base date: {RESULT['metricBaseDate']}")
print(f"Pipeline version: {RESULT['pipelineVersion']}")
print(f"Fixture package ready: {RESULT['fixturePackageReady']}")
print(f"Production publish ready: {RESULT['productionPublishReady']}")
print(f"App export approved: {RESULT['appExportApproved']}")
print(f"Source adapter summary: {RESULT['outputs']['sourceAdapterSummaryJson']}")
print(f"Source adapter checkpoint: {RESULT['outputs']['sourceAdapterCheckpointJson']}")
for key, value in RESULT["summary"].items():
    print(f"{key}: {value}")
print(f"Output ZIP: {RESULT['outputs']['zipPackage']}")

## 5. Download output ZIP

Download the ZIP after checking the audit report and manifest. This ZIP is a fixture output package, not production app approval.

In [ ]:
zip_path = Path(RESULT["outputs"]["zipPackage"])
print(zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Colab download helper is unavailable in this runtime. Open the ZIP path above.")